### Implement temporal Fusion Transformer

In [17]:
import pandas as pd
import numpy as np
import torch
from pytorch_lightning import Trainer
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, Baseline
from pytorch_forecasting.data import NaNLabelEncoder
from pytorch_forecasting.metrics import QuantileLoss

In [26]:
# Prepare Sample Data
# Generate synthetic dataset
df = pd.DataFrame()
df["time_idx"] = np.tile(np.arange(100), 10)
df["group"] = np.repeat(np.arange(10), 100)
df["value"] = np.random.rand(1000) + df["group"] * 0.1

df["group"] = df["group"].astype(str)
# Add a future covariate
df["month"] = df["time_idx"] % 12

# 📊 Define dataset parameters
max_encoder_length = 30
max_prediction_length = 10

# 🧹 Create TimeSeriesDataSet
training = TimeSeriesDataSet(
    df[df.time_idx < 90],
    time_idx="time_idx",
    target="value",
    group_ids=["group"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["group"],
    time_varying_known_reals=["time_idx", "month"],
    time_varying_unknown_reals=["value"],
    target_normalizer=NaNLabelEncoder(),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

# 🧪 Validation dataset
validation = TimeSeriesDataSet.from_dataset(training, df[df.time_idx >= 90], stop_randomization=True)

# 🔄 Dataloaders
train_dataloader = training.to_dataloader(train=True, batch_size=64, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=64, num_workers=0)

# 🧠 Build the model
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=16,
    attention_head_size=1,
    dropout=0.1,
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=3,
    log_val_interval=1,
)

# 👀 Print number of parameters
print(f"Number of parameters: {tft.size()/1e3:.1f}k")

# 🚀 Train the model
early_stop = EarlyStopping(monitor="val_loss", patience=5, min_delta=1e-4)
lr_logger = LearningRateMonitor()

trainer = Trainer(
    max_epochs=5,
    gradient_clip_val=0.1,
    callbacks=[early_stop, lr_logger],
    enable_model_summary=True,
)

trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

# 🔮 Predict on validation set
actuals = torch.cat([y[0] for x, y in iter(val_dataloader)])
predictions = tft.predict(val_dataloader)

# ✅ Done! Check shapes
print(f"Predictions shape: {predictions.shape}")

KeyError: "Unknown category '0.13910569992770805' encountered. Set `add_nan=True` to allow unknown categories"

In [19]:
df

,time_idx,group,value,month
0,0,0,0.184218,0
1,1,0,0.406191,1
2,2,0,0.123118,2
3,3,0,0.366734,3
4,4,0,0.324108,4
...,...,...,...,...
995,95,9,1.754860,11
996,96,9,1.706727,0
997,97,9,1.047875,1
998,98,9,1.619819,2


In [20]:
# Create TimeSeriesDataSet
max_encoder_length = 30
max_prediction_length = 10

training = TimeSeriesDataSet(
    df[df.time_idx < 90],
    time_idx="time_idx",
    target="value",
    group_ids=["group"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["group"],
    time_varying_known_reals=["time_idx", "month"],
    time_varying_unknown_reals=["value"],
    target_normalizer=NaNLabelEncoder()
)


In [21]:
# Create DataLoader
train_dataloader = training.to_dataloader(train=True, batch_size=32, num_workers=0)

In [22]:
# Define the model
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=16,
    attention_head_size=1,
    dropout=0.1,
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=4,
)


/Users/rattanak/miniconda3/envs/py312/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/Users/rattanak/miniconda3/envs/py312/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [23]:
# Train the model
trainer = Trainer(max_epochs=5, gradient_clip_val=0.1)
trainer.fit(tft, train_dataloaders=train_dataloader)
# lightning_module = tft.to_lightning_module()
# trainer.fit(lightning_module, train_dataloaders=train_dataloader)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


TypeError: `model` must be a `LightningModule` or `torch._dynamo.OptimizedModule`, got `TemporalFusionTransformer`